# A model registry and promoting versions

The parcel-locker team's damage classifier, now with a real MLflow model registry. You will register three versions, release one by moving the `champion` alias, roll it back, and then find why Monday's roll-back did not reach the service.

About 10 minutes. Everything runs on a free Colab CPU runtime. The registry is a small sqlite file in this notebook, so you need no server and no account.

## 1. Install MLflow

This installs the same MLflow version the lesson used. `mlflow-skinny` is MLflow without the web UI. `alembic` and `sqlalchemy` let it keep the registry in a sqlite file.

**Look for:** no output is fine. Colab may say a restart is not needed, or show nothing.

In [1]:
%pip install -q mlflow-skinny==3.18.0 alembic==1.20.1 sqlalchemy==2.1.8

Note: you may need to restart the kernel to use updated packages.


## 2. The classifier and the data

This is the lesson's stand-in classifier. Each photo is already reduced to three scores between 0 and 1: crushed, torn, wet. A version is a set of weights plus a threshold: the photo is `damaged` when the weighted total is above the threshold. There is no randomness, so your labels and metrics match the lesson. Run ids and times will differ.

The cell also starts a fresh registry in the folder `registry-work`.

**Look for:** the tracking URI that starts with `sqlite:///`, and the six photo ids. MLflow may also print a hint to "Add type hints to the `predict` method". It is only a suggestion; ignore it.

In [2]:
import os, shutil, logging
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"   # must be set before mlflow is imported

import mlflow
import numpy as np
from mlflow import MlflowClient

logging.getLogger("mlflow").setLevel(logging.ERROR)   # hide MLflow's info and warning lines
logging.getLogger("alembic").setLevel(logging.ERROR)

NAME = "parcel-damage"

# the three versions the team trained: weights for (crushed, torn, wet), and the threshold
VERSIONS = {
    1: {"weights": [1.0, 1.0, 0.0], "threshold": 0.9, "note": "first model, ignores water"},
    2: {"weights": [1.0, 1.0, 1.0], "threshold": 0.9, "note": "adds the wet score"},
    3: {"weights": [1.0, 1.0, 1.0], "threshold": 0.6, "note": "lower threshold, catch more damage"},
}

# a small labelled check set: (crushed, torn, wet), really damaged?
CHECK = [
    ([0.80, 0.30, 0.00], True), ([0.10, 0.70, 0.30], True), ([0.20, 0.10, 0.80], True),
    ([0.30, 0.20, 0.30], True), ([0.05, 0.05, 0.10], False), ([0.30, 0.10, 0.10], False),
    ([0.20, 0.30, 0.20], False), ([0.10, 0.10, 0.50], False), ([0.40, 0.00, 0.20], False),
    ([0.00, 0.10, 0.05], False),
]

# the weekend's photos at one locker bank
PHOTOS = {
    "P-101": [0.05, 0.10, 0.05], "P-102": [0.40, 0.10, 0.20], "P-103": [0.70, 0.30, 0.10],
    "P-104": [0.10, 0.05, 0.60], "P-105": [0.20, 0.20, 0.10], "P-106": [0.10, 0.60, 0.40],
}
photos = np.array(list(PHOTOS.values()))


class DamageModel(mlflow.pyfunc.PythonModel):
    def __init__(self, weights, threshold):
        self.weights = np.array(weights)
        self.threshold = threshold

    def predict(self, context, model_input, params=None):
        scores = np.asarray(model_input, dtype=float) @ self.weights
        return ["damaged" if s > self.threshold else "fine" for s in scores]


WORK = "registry-work"
shutil.rmtree(WORK, ignore_errors=True)       # start from an empty registry each time
os.makedirs(WORK)
mlflow.set_tracking_uri("sqlite:///" + WORK + "/registry.db")
mlflow.set_experiment("parcel-damage")
client = MlflowClient()

print("tracking URI:", mlflow.get_tracking_uri())
print("photos:", list(PHOTOS))

tracking URI: sqlite:///registry-work/registry.db
photos: ['P-101', 'P-102', 'P-103', 'P-104', 'P-105', 'P-106']


## 3. Train and log three versions

Each version is scored on the check set of 10 photos (4 really damaged) and logged as a run with `log_model`, as in the last lesson. Nothing is registered yet: a logged model only has a `models:/m-...` address.

**Look for:** recall rises from 0.25 to 0.75 to 1.00, but version 3 pays with precision 0.57 and 3 false alarms. This cell takes about half a minute.

In [3]:
def log_version(n):
    v = VERSIONS[n]
    model = DamageModel(v["weights"], v["threshold"])
    said = model.predict(None, [x for x, _ in CHECK])
    tp = sum(1 for s, (_, d) in zip(said, CHECK) if s == "damaged" and d)
    fp = sum(1 for s, (_, d) in zip(said, CHECK) if s == "damaged" and not d)
    fn = sum(1 for s, (_, d) in zip(said, CHECK) if s == "fine" and d)
    with mlflow.start_run(run_name=f"train-{n}") as run:
        mlflow.log_params({"weights": v["weights"], "threshold": v["threshold"]})
        mlflow.log_metrics({"recall": tp / (tp + fn), "precision": tp / (tp + fp)})
        info = mlflow.pyfunc.log_model(name="model", python_model=model)
    print(f"[train] {n}: run_id={run.info.run_id} recall={tp / (tp + fn):.2f} precision={tp / (tp + fp):.2f} "
          f"false_alarms={fp} model_uri={info.model_uri}")
    return info.model_uri, run.info.run_id


logged = {n: log_version(n) for n in VERSIONS}
uris = {n: uri for n, (uri, _) in logged.items()}
RUN_OF = {run_id: n for n, (_, run_id) in logged.items()}   # run id -> version number, used later

[train] 1: run_id=9491b08dc3c843a4ae24a06a542a4a53 recall=0.25 precision=1.00 false_alarms=0 model_uri=models:/m-26cb440c226344469fac0b02d02a1f93


[train] 2: run_id=d2b6d734a08e4e3bb95eebacac7bf358 recall=0.75 precision=1.00 false_alarms=0 model_uri=models:/m-7cdb2b15761b4526a4a7d1308613bd66


[train] 3: run_id=382d1d14fe534eb8852b967a49fc0ebd recall=1.00 precision=0.57 false_alarms=3 model_uri=models:/m-04326ed5019e41d785aa9dd341861d3b


## 4. Earlier in the week: versions 1 and 2

Before Friday, the team registered version 1, then version 2, and moved `champion` each time. This is the state the lesson's screen 4 starts from.

**Look for:** "Successfully registered model 'parcel-damage'" once, then "Created version '2'", and champion ending on 2.

In [4]:
for n in (1, 2):
    mv = mlflow.register_model(uris[n], NAME)
    client.set_registered_model_alias(NAME, "champion", mv.version)
    print(f"[earlier] registered version {mv.version!r}, champion -> {mv.version}")

[earlier] registered version 1, champion -> 1
[earlier] registered version 2, champion -> 2


Successfully registered model 'parcel-damage'.
Created version '1' of model 'parcel-damage'.
Registered model 'parcel-damage' already exists. Creating a new version of this model...
Created version '2' of model 'parcel-damage'.


## 5. Friday's release and the roll-back

This is the function `release` from the lesson, line for line. It registers version 3, checks where champion points, moves champion to 3, loads the service by alias, rolls back to 2, and asks the **same** `service` object again. The last line loads a fresh copy.

**Look for:**
- "Created version '3'" and then "champion is still 2": registering is not releasing.
- "service says" and "after roll-back, same service says" print the same list, with 4 damaged labels. The loaded model stays in memory.
- "reloaded:" has only 2 damaged labels. P-102 and P-104 turn fine.

In [5]:
def release(model_uri, photos):
    mv = mlflow.register_model(model_uri, "parcel-damage")
    print("registered version", mv.version)
    champ = client.get_model_version_by_alias("parcel-damage", "champion")
    print("champion is still", champ.version)
    client.set_registered_model_alias("parcel-damage", "champion", mv.version)
    service = mlflow.pyfunc.load_model("models:/parcel-damage@champion")
    print("service says", service.predict(photos))
    client.set_registered_model_alias("parcel-damage", "champion", "2")
    print("after roll-back, same service says", service.predict(photos))
    print("reloaded:", mlflow.pyfunc.load_model("models:/parcel-damage@champion").predict(photos))


release(uris[3], photos)

registered version 3
champion is still 2
service says ['fine', 'damaged', 'damaged', 'damaged', 'fine', 'damaged']
after roll-back, same service says ['fine', 'damaged', 'damaged', 'damaged', 'fine', 'damaged']
reloaded: ['fine', 'fine', 'damaged', 'fine', 'fine', 'damaged']


Registered model 'parcel-damage' already exists. Creating a new version of this model...
Created version '3' of model 'parcel-damage'.


## 6. Ask the registry what is live

Every version stays in the registry, each linked to the run that trained it. `get_model_version_by_alias` answers "which version is champion?" in one call.

**Look for:** only version 2 lists `champion`; versions 1 and 3 have no alias. All three are `READY`.

In [6]:
# search_model_versions does not fill in aliases on this store, so read each version
for found in sorted(client.search_model_versions(f"name='{NAME}'"), key=lambda m: int(m.version)):
    mv = client.get_model_version(NAME, str(found.version))
    print(f"version={mv.version!r} aliases={mv.aliases} run_id={mv.run_id} status={mv.status}")
champ = client.get_model_version_by_alias(NAME, "champion")
print("get_model_version_by_alias ->", type(champ).__name__, "version", repr(champ.version), "aliases", champ.aliases)

version=1 aliases=[] run_id=9491b08dc3c843a4ae24a06a542a4a53 status=READY
version=2 aliases=['champion'] run_id=d2b6d734a08e4e3bb95eebacac7bf358 status=READY
version=3 aliases=[] run_id=382d1d14fe534eb8852b967a49fc0ebd status=READY
get_model_version_by_alias -> ModelVersion version 2 aliases ['champion']


## 7. A second alias and a pinned address

Put `challenger` on version 3: a version can carry several aliases, and champion does not move. Then load `models:/parcel-damage/3`, a **pinned** address. It always means version 3, wherever the aliases point.

**Look for:** version 2 has `['champion']`, version 3 has `['challenger']`. The pinned load prints version 3's labels (4 damaged), even though champion is on 2.

In [7]:
client.set_registered_model_alias(NAME, "challenger", "3")
for n in VERSIONS:
    print(f"[challenger] version={n} aliases={client.get_model_version(NAME, str(n)).aliases}")

pinned = mlflow.pyfunc.load_model(f"models:/{NAME}/3")
print("[pinned] models:/parcel-damage/3 says", pinned.predict(photos))

[challenger] version=1 aliases=[]


[challenger] version=2 aliases=['champion']
[challenger] version=3 aliases=['challenger']
[pinned] models:/parcel-damage/3 says ['fine', 'damaged', 'damaged', 'damaged', 'fine', 'damaged']


## 8. Alias names MLflow refuses

You choose alias names, but MLflow refuses `v` plus a number (it would look like a version) and `latest` in any case. This cell is **meant** to hit errors: each call is wrapped in `try/except` and the error is printed.

**Look for:** three refusals, and the name `stable` accepted.

In [8]:
for alias in ("v3", "latest", "Latest", "stable"):
    try:
        client.set_registered_model_alias(NAME, alias, "3")
        print(f"[alias {alias}] accepted")
    except Exception as e:  # we want to print what MLflow says
        print(f"[alias {alias}]", type(e).__name__, str(e).splitlines()[0][:200])
client.delete_registered_model_alias(NAME, "stable")   # tidy up: keep only champion and challenger

[alias v3] MlflowException Version alias name 'v3' is reserved.
[alias latest] MlflowException 'latest' alias name (case insensitive) is reserved.
[alias Latest] MlflowException 'latest' alias name (case insensitive) is reserved.
[alias stable] accepted


## 9. Record why the alias moved

An alias shows only where it points **now**. A tag on the version keeps the reason for the roll-back with it.

**Look for:** version 2's tags include `rolled_back_from: '3'`.

In [9]:
client.set_model_version_tag(NAME, "2", "rolled_back_from", "3")
print("[tag] version 2 tags", client.get_model_version(NAME, "2").tags)

[tag] version 2 tags {'rolled_back_from': '3'}


## 10. Monday: the roll-back that changed nothing

The locker service picks its model from the environment variable `MODEL_URI`, with the champion alias as the default. Someone put a pinned address in the container's settings, `deploy.env`, "to be sure version 3 is live".

The cell starts a fresh container: it loads the settings, moves champion to 2, and restarts the service.

**Look for:** the log says "roll-back: champion -> version 2", yet the service logs `loaded models:/parcel-damage/3` with version 3's run id, and P-102 and P-104 are still `damaged`. The last line names the version from the run id.

In [10]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
ops = logging.getLogger("ops")


def start_service():
    # the service code: read MODEL_URI from the environment, with the alias as the default
    uri = os.environ.get("MODEL_URI", "models:/parcel-damage@champion")
    model = mlflow.pyfunc.load_model(uri)
    logging.getLogger("locker-service").info("loaded %s run_id=%s", uri, model.metadata.run_id)
    return model


def monday(deploy_env):
    os.environ.pop("MODEL_URI", None)        # a new container starts with a clean environment
    for line in deploy_env.strip().splitlines():
        key, _, value = line.strip().partition("=")
        ops.info("env %s=%s", key, value)
        os.environ[key] = value
    client.set_registered_model_alias(NAME, "champion", "2")
    champ = client.get_model_version_by_alias(NAME, "champion")
    ops.info("roll-back: champion -> version %s run_id=%s", champ.version, champ.run_id)
    ops.info("restarting locker-service")
    model = start_service()
    for pid, label in zip(PHOTOS, model.predict(photos)):
        logging.getLogger("locker-service").info("%s -> %s", pid, label)
    print("the service is running version", RUN_OF[model.metadata.run_id], "| champion is on version", champ.version)


DEPLOY_ENV = '''
LOCKER_BANK=north-7
MODEL_URI=models:/parcel-damage/3
LOG_LEVEL=INFO
'''
monday(DEPLOY_ENV)

2026-11-14 15:29:20,507 INFO ops: env LOCKER_BANK=north-7


2026-11-14 15:29:20,508 INFO ops: env MODEL_URI=models:/parcel-damage/3


2026-11-14 15:29:20,508 INFO ops: env LOG_LEVEL=INFO


2026-11-14 15:29:20,587 INFO ops: roll-back: champion -> version 2 run_id=d2b6d734a08e4e3bb95eebacac7bf358


2026-11-14 15:29:20,587 INFO ops: restarting locker-service


2026-11-14 15:29:20,599 INFO locker-service: loaded models:/parcel-damage/3 run_id=382d1d14fe534eb8852b967a49fc0ebd


... (3 outputs left out)

2026-11-14 15:29:20,600 INFO locker-service: P-104 -> damaged


2026-11-14 15:29:20,601 INFO locker-service: P-105 -> fine


2026-11-14 15:29:20,601 INFO locker-service: P-106 -> damaged


the service is running version 3 | champion is on version 2


## 11. The fix: drop the override

`os.environ.get` uses the default only when the variable is missing. Remove the `MODEL_URI` line, so the alias decides, and restart.

**Look for:** `loaded models:/parcel-damage@champion` with version 2's run id, and P-102 and P-104 now `fine`.

In [11]:
DEPLOY_FIXED_ENV = '''
LOCKER_BANK=north-7
LOG_LEVEL=INFO
'''
monday(DEPLOY_FIXED_ENV)

2026-11-14 15:29:20,605 INFO ops: env LOCKER_BANK=north-7


2026-11-14 15:29:20,605 INFO ops: env LOG_LEVEL=INFO


2026-11-14 15:29:20,610 INFO ops: roll-back: champion -> version 2 run_id=d2b6d734a08e4e3bb95eebacac7bf358


2026-11-14 15:29:20,611 INFO ops: restarting locker-service


2026-11-14 15:29:20,623 INFO locker-service: loaded models:/parcel-damage@champion run_id=d2b6d734a08e4e3bb95eebacac7bf358


2026-11-14 15:29:20,624 INFO locker-service: P-101 -> fine


... (2 outputs left out)

2026-11-14 15:29:20,624 INFO locker-service: P-104 -> fine


2026-11-14 15:29:20,625 INFO locker-service: P-105 -> fine


2026-11-14 15:29:20,625 INFO locker-service: P-106 -> damaged


the service is running version 2 | champion is on version 2


## Now change it

Each cell below has one line marked `# <- edit`. Change it, run the cell, and compare with what you saw above.

**Experiment 1: which version does the pin win against?** Change `/3` to `/1`. The service should log version 1's run id and say the service runs version 1, while champion stays on 2. A pinned number in the settings beats the alias every time, whatever number it is.

In [12]:
PIN = "models:/parcel-damage/3"   # <- edit: try models:/parcel-damage/1
monday(f'''
LOCKER_BANK=north-7
MODEL_URI={PIN}
LOG_LEVEL=INFO
''')

2026-11-14 15:29:20,633 INFO ops: env LOCKER_BANK=north-7


2026-11-14 15:29:20,634 INFO ops: env MODEL_URI=models:/parcel-damage/3


2026-11-14 15:29:20,635 INFO ops: env LOG_LEVEL=INFO


2026-11-14 15:29:20,643 INFO ops: roll-back: champion -> version 2 run_id=d2b6d734a08e4e3bb95eebacac7bf358


2026-11-14 15:29:20,643 INFO ops: restarting locker-service


2026-11-14 15:29:20,652 INFO locker-service: loaded models:/parcel-damage/3 run_id=382d1d14fe534eb8852b967a49fc0ebd


... (3 outputs left out)

2026-11-14 15:29:20,653 INFO locker-service: P-104 -> damaged


2026-11-14 15:29:20,653 INFO locker-service: P-105 -> fine


2026-11-14 15:29:20,654 INFO locker-service: P-106 -> damaged


the service is running version 3 | champion is on version 2


**Experiment 2: load by another alias.** Change `"champion"` to `"challenger"`. The labels should go back to version 3's (4 damaged), because challenger points at version 3. The service code is the same; only the alias name changed.

In [13]:
ALIAS = "champion"   # <- edit: try "challenger"
m = mlflow.pyfunc.load_model(f"models:/parcel-damage@{ALIAS}")
print(f"models:/parcel-damage@{ALIAS} is version", RUN_OF[m.metadata.run_id], "and says", m.predict(photos))

models:/parcel-damage@champion is version 2 and says ['fine', 'fine', 'damaged', 'fine', 'fine', 'damaged']


**Experiment 3: a new version is not a release.** This registers a version 4 with threshold 0.75. On this small check set it catches all 4 damaged parcels with no false alarm, and still champion stays on version 2. Change `0.75` to `0.5` or `1.2`: the catches and false alarms change, but champion never moves by itself. Only setting the alias releases a version.

In [14]:
THRESHOLD = 0.75   # <- edit: try 0.5 or 1.2
model = DamageModel([1.0, 1.0, 1.0], THRESHOLD)
said = model.predict(None, [x for x, _ in CHECK])
caught = sum(1 for s, (_, d) in zip(said, CHECK) if s == "damaged" and d)
false_alarms = sum(1 for s, (_, d) in zip(said, CHECK) if s == "damaged" and not d)
with mlflow.start_run(run_name="train-new"):
    info = mlflow.pyfunc.log_model(name="model", python_model=model)
mv = mlflow.register_model(info.model_uri, NAME)
print(f"version {mv.version}: threshold={THRESHOLD} caught {caught} of 4 damaged, false alarms {false_alarms}")
print("champion is on version", client.get_model_version_by_alias(NAME, "champion").version)

version 4: threshold=0.75 caught 4 of 4 damaged, false alarms 0
champion is on version 2


Registered model 'parcel-damage' already exists. Creating a new version of this model...
Created version '4' of model 'parcel-damage'.


## What you should see

- `register_model` prints "Created version '3' of model 'parcel-damage'.", and right after it champion still points at version 2.
- The model loaded before the roll-back prints the same 4 damaged labels after champion moves back to 2; a fresh load prints 2 damaged labels.
- After the roll-back, only version 2 lists `champion` in its aliases; version 3 has none.
- Setting an alias named `v3` raises `MlflowException Version alias name 'v3' is reserved.`
- With `MODEL_URI=models:/parcel-damage/3` set, the restarted service logs version 3's run id; without it, version 2's.